# Import the Data

Import the CSV file (from CaImAn) containing the cell trace data. Import it into a dataframe. Since the data is organized as rows rather than columns, I first convert the data so that each cell is a column, with the column headers corresponding with the cells. 

In [ ]:
import pandas as pd
import numpy as np
import scipy as sy
import statistics as st
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from scipy.signal import savgol_filter

In [ ]:
raw_data = pd.read_csv('example_data.csv',header=0)
raw_data

# Apply gaussian filter to each column

In [ ]:

cols = raw_data.columns
#cols=["Cell 35"]
filtered_data = pd.DataFrame()

for col in cols:
    filtering = gaussian_filter1d(raw_data[col], 3)
    filtered_data[col] = filtering

filtered_data.head()

## Let's plot the gaussian filtered cells

Let's go through and plot all of our cells so that we can visualize the traces

%matplotlib inline

for col in filtered_data.columns:
    plt.plot(filtered_data[col], color = 'blue')
    plt.title(col)
    plt.show()

## Normalize each trace (bring the min to 0)

Baseline of traces may be greater than zero. This may alter the measurement of peak amplitudes. Here, normalize each trace by subtracting the entire trace by its minimum value. 

In [ ]:
for col in cols:
#     index = raw_data[col].idxmin()
#     min_med = st.median(raw_data[col].loc[index-10:index+10])
#     raw_data[col] = raw_data[col].sub(min_med)
    min_col = np.min(filtered_data[col])
    
    if (min_col > 0):
        filtered_data[col] = filtered_data[col] - abs(min_col)

    else:
        filtered_data[col] = filtered_data[col] + abs(min_col)

# Find Peaks in the data

Creating a function that will iterate through each column of the dataframe (each cell) and find the peaks within the data. The number of peaks, average peak height, and the index of each peak will be appeneded to a list, giving us these values for each cell in the dataframe. The function will return each of these. 

In [ ]:
def get_peaks(data,standard_data):
    peak_num = []
    avg_peak_amp = []
    max_peak_amp = []
    peak_index = []
    empty_cells = []
    
    cols = data.columns
    
    for col in cols:
        std = standard_data[col].std()
        peaks, properties = find_peaks(data[col], distance=35, height=(np.mean(standard_data[col])+1.0*std), prominence=15)
        if len(peaks) == 0:
            empty_cells.append(col)
            pass
        else:
            peak_num.append(len(peaks))
            avg_peak_amp.append(properties["peak_heights"].mean())
            max_peak_amp.append(properties['peak_heights'].max())
            peak_index.append(peaks)
        
    return peak_num, avg_peak_amp, peak_index, max_peak_amp, empty_cells


In [ ]:
raw_data_first = filtered_data.iloc[0:242]
raw_data_second = filtered_data.iloc[363:726]
Half_Num=1

if Half_Num==1:
    data_half = raw_data_first
else:
    data_half = raw_data_second

In [ ]:
#get_peaks(data_half,filtered_data)

peak_num,avg_peak_amp,peak_index,max_peak_amp,empty_cells = get_peaks(data_half,filtered_data)
if Half_Num==2:
    peak_index = [x+363 for x in peak_index]
overall_avg_amp_avg = np.mean(avg_peak_amp)
overall_max_amp_avg = np.mean(max_peak_amp)
sum_peak_num = sum(peak_num)

# Visualize the locations of the selected peaks

Plotting the traces for each cell with markers at the identified peaks. This will allow us to change the parameters of the find peak function so that it correctly identifies the peaks. 

First create a dictionary with the peak locations for each of the columns (cells) of the original dataframe. The column names (cell name) will be the dictionary key, and the array of peak indicies will be the values.

In [ ]:
filtered_cols = [e for e in cols if e not in empty_cells]

In [ ]:
empty_cells

In [ ]:
data_half[col]

In [ ]:


# Create a zip object from two lists
zip_peaks = zip(filtered_cols, peak_index)
 
# Create a dictionary from zip object
peaks_dict = dict(zip_peaks)

Now plot the cell traces with the peaks marked. For each column (cell) individually plot the data trace. On the same plot, also plot the peak locations with a red 'x'. 

In [ ]:
for col in cols:
    print(col)
    plt.plot(data_half[col], color = 'blue')
    plt.ylim(-20,max(filtered_data[col])*1.05)
    if col not in empty_cells:
        plt.plot(peaks_dict[col], data_half[col][peaks_dict[col]], "x", color = 'red')
    plt.title(col)
    plt.show()


In [ ]:
import numpy
from statistics import mean

area_cells = []

for col in cols:
    area_cells.append(np.trapz(data_half[col]))
    
print("Total # of Domain: " + str(len(filtered_cols)))
print("Average area under the curve: " + str(mean(area_cells)))
print("Total # of area under the curve: " + str(sum(area_cells)))
print("Average # of peaks: " + str(mean(peak_num)))
print("Total # of peaks: " + str(sum_peak_num))
print("Average peak amplitude: " + str(overall_avg_amp_avg))
print("Average maximum peak amplitude: " + str(overall_max_amp_avg))


In [ ]:
peak_num

In [ ]:
overall_avg_amp_avg